# 04 — Calibrate + export (ONNX int8)

Calibrates `student-v1` (single global temperature, fit on held-out `val_en`), exports it to ONNX (opset 17), quantizes to int8 (`quantize_dynamic`), runs a parity check against the fp32 PyTorch model, then uploads the versioned artifact set to R2.

**Runtime:** Colab, CPU is fine — no training happens here, just forward passes and export/quantization.

## Setup

In [ ]:
# Public repo, no auth needed
!git clone --depth 1 https://github.com/gjvarun0307/real-time-moderation-pipeline.git /content/repo

import sys

sys.path.insert(0, "/content/repo/src")

In [ ]:
# install requirements
!pip install -q "transformers>=4.51" "sentencepiece>=0.2" "scikit-learn>=1.5" \
    "pyarrow>=17.0" "homoglyphs>=2.0" "regex>=2024.5.15" "onnx>=1.16" \
    "onnxruntime>=1.18" "boto3>=1.34"

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

DATA_DIR = Path("/content/drive/MyDrive/moderation-pipeline/data/processed")
CHECKPOINT_ROOT = Path("/content/drive/MyDrive/moderation-pipeline/checkpoints")
STUDENT_CHECKPOINT_DIR = CHECKPOINT_ROOT / "student-v1"
EXPORT_DIR = CHECKPOINT_ROOT / "export-v1"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

## Config

In [ ]:
import subprocess

import torch

MAX_SEQ_LEN = 192  # must match training
PARITY_SAMPLE_SIZE = 5000
PARITY_MAX_DROP = 0.01
ONNX_OPSET = 17
EXPORT_VERSION = 1
SEED = 42

# change cuda or cpu
DEVICE = torch.device("cpu")
torch.manual_seed(SEED)

# R2 (Cloudflare) — bucket + non-secret credentials. The secret key never
# goes in this notebook: it's read from Colab's own Secrets manager (key
# icon, left sidebar) at upload time.
R2_BUCKET = "moderation-pipeline"
R2_ACCOUNT_ID = "987b06fd7083dcd8e6e210c4afdedf53"
R2_ENDPOINT = f"https://{R2_ACCOUNT_ID}.r2.cloudflarestorage.com"
R2_ACCESS_KEY_ID = "3c1b25cb63cc6454ec12dc93c8222185"

GIT_SHA = subprocess.run(
    ["git", "-C", "/content/repo", "rev-parse", "--short", "HEAD"],
    capture_output=True,
    text=True,
    check=True,
).stdout.strip()
VERSION_TAG = f"v{EXPORT_VERSION}-{GIT_SHA}"
print(f"exporting {VERSION_TAG}")

## Load student checkpoint + held-out data

In [ ]:
import pandas as pd
from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification, AutoTokenizer

from training.teacher_model import ToxicityDataset, make_collate_fn

tokenizer = AutoTokenizer.from_pretrained(STUDENT_CHECKPOINT_DIR)
model = AutoModelForSequenceClassification.from_pretrained(STUDENT_CHECKPOINT_DIR).to(
    DEVICE
)
model.eval()

collate_fn = make_collate_fn(tokenizer, MAX_SEQ_LEN)
val_df = pd.read_parquet(DATA_DIR / "val_en.parquet")
val_ds = ToxicityDataset(val_df)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, collate_fn=collate_fn)
print(f"val_en: {len(val_df)} rows")

In [ ]:
all_logits, all_labels = [], []
with torch.no_grad():
    for batch in val_loader:
        labels = batch.pop("labels")
        logits = model(**batch).logits
        all_logits.append(logits)
        all_labels.append(labels)
val_logits = torch.cat(all_logits)
val_labels = torch.cat(all_labels)
print(f"collected {val_logits.shape[0]} val_en logits for calibration")

## Calibrate: fit temperature on held-out val_en

In [ ]:
from training.calibration import compute_ece, fit_temperature

probs_uncalibrated = torch.sigmoid(val_logits).cpu().numpy()
ece_before = compute_ece(probs_uncalibrated, val_labels.cpu().numpy())

temperature = fit_temperature(val_logits, val_labels)

probs_calibrated = torch.sigmoid(val_logits / temperature).cpu().numpy()
ece_after = compute_ece(probs_calibrated, val_labels.cpu().numpy())

print(f"temperature: {temperature:.4f}")
print(f"ECE before calibration: {ece_before:.4f}")
print(f"ECE after calibration:  {ece_after:.4f}")

## Export to ONNX (opset 17)

In [ ]:
class _LogitsOnly(torch.nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.base_model = base_model

    def forward(self, input_ids, attention_mask):
        return self.base_model(input_ids=input_ids, attention_mask=attention_mask).logits


onnx_export_model = _LogitsOnly(model).eval()

onnx_fp32_path = EXPORT_DIR / "model_fp32.onnx"
dummy_input_ids = torch.zeros((1, MAX_SEQ_LEN), dtype=torch.long, device=DEVICE)
dummy_attention_mask = torch.ones((1, MAX_SEQ_LEN), dtype=torch.long, device=DEVICE)

torch.onnx.export(
    onnx_export_model,
    (dummy_input_ids, dummy_attention_mask),
    str(onnx_fp32_path),
    input_names=["input_ids", "attention_mask"],
    output_names=["logits"],
    dynamic_axes={
        "input_ids": {0: "batch", 1: "sequence"},
        "attention_mask": {0: "batch", 1: "sequence"},
        "logits": {0: "batch"},
    },
    opset_version=ONNX_OPSET,
    dynamo=False,  # the new dynamo=True default mis-handles dynamic_axes here
)
print(f"exported fp32 ONNX model to {onnx_fp32_path}")

## Quantize to int8 (`quantize_dynamic`)

In [ ]:
from onnxruntime.quantization import QuantType, quantize_dynamic

onnx_int8_path = EXPORT_DIR / "model.onnx"
quantize_dynamic(
    model_input=str(onnx_fp32_path),
    model_output=str(onnx_int8_path),
    weight_type=QuantType.QInt8,
)

fp32_mb = onnx_fp32_path.stat().st_size / 1e6
int8_mb = onnx_int8_path.stat().st_size / 1e6
print(f"fp32 ONNX: {fp32_mb:.1f} MB")
print(f"int8 ONNX: {int8_mb:.1f} MB ({100 * (1 - int8_mb / fp32_mb):.1f}% smaller)")

## Parity check: fp32 PyTorch vs. int8 ONNX

In [ ]:
import onnxruntime as ort

from training.calibration import OnnxRuntimeAdapter, parity_ok
from training.teacher_model import evaluate

parity_df = val_df.sample(
    n=min(PARITY_SAMPLE_SIZE, len(val_df)), random_state=SEED
).reset_index(drop=True)
parity_ds = ToxicityDataset(parity_df)
parity_loader = DataLoader(parity_ds, batch_size=64, shuffle=False, collate_fn=collate_fn)

fp32_scores = evaluate(model, parity_loader, DEVICE)

session = ort.InferenceSession(str(onnx_int8_path), providers=["CPUExecutionProvider"])
int8_scores = evaluate(OnnxRuntimeAdapter(session), parity_loader, DEVICE)

parity_passed, deltas = parity_ok(fp32_scores, int8_scores, max_drop=PARITY_MAX_DROP)

print("fp32 PyTorch:", fp32_scores)
print("int8 ONNX:   ", int8_scores)
print("deltas (int8 - fp32):", deltas)
print(f"\nparity check {'PASSED' if parity_passed else 'FAILED'} (max allowed drop: {PARITY_MAX_DROP})")
assert parity_passed, f"parity check failed: {deltas}"

## Write artifacts: tokenizer/, calibration.json, thresholds.yaml, model_card.md

In [ ]:
import json

tokenizer_dir = EXPORT_DIR / "tokenizer"
tokenizer.save_pretrained(tokenizer_dir)

calibration = {
    "temperature": temperature,
    "fitted_on": "val_en",
    "method": "single global temperature scaling (Guo et al. 2017)",
    "ece_before": ece_before,
    "ece_after": ece_after,
}
with open(EXPORT_DIR / "calibration.json", "w") as f:
    json.dump(calibration, f, indent=2)

In [ ]:
# eval/thresholds.yaml regression gate values. MHC pass rate and
# attack success rate aren't measurable yet -- eval/adversarial/ isn't built --
# recorded here as the target this export version is meant to be judged against.
thresholds = {
    "min_pr_auc_toxic": 0.86,
    "max_fpr_at_tpr90_global": 0.08,
    "max_fpr_at_tpr90_per_lang": 0.18,
    "min_mhc_pass_rate": 0.70,
    "max_attack_success_rate": 0.25,
    "max_ece": 0.05,
}
with open(EXPORT_DIR / "thresholds.yaml", "w") as f:
    for key, value in thresholds.items():
        f.write(f"{key}: {value}\n")

In [ ]:
with open(STUDENT_CHECKPOINT_DIR / "metrics.json") as f:
    train_metrics = json.load(f)

model_card_lines = [
    f"# Moderation classifier — {VERSION_TAG}",
    "",
    f"6-layer distilled `xlm-roberta-base` student "
    f"({train_metrics['student_param_count']:,} params, quantized to int8 ONNX), "
    f"trained via knowledge distillation from a fine-tuned `xlm-roberta-base` teacher "
    f"({train_metrics['teacher_param_count']:,} params) on the English Jigsaw Toxic "
    "Comment dataset. Multi-label: toxic, severe_toxic, obscene, threat, insult, "
    "identity_hate.",
    "",
    "## Training data",
    "",
    "English only (Jigsaw Toxic Comment Classification Challenge, merged train+test). "
    "No non-English training data — es/it/tr/ja performance below is zero-shot "
    "cross-lingual transfer, not fine-tuned.",
    "",
    "## Metrics",
    "",
    "| metric | value |",
    "|---|---|",
    f"| val_en toxic PR-AUC | {train_metrics['val_en_pr_auc']['toxic']:.4f} |",
    f"| val_en macro PR-AUC | {train_metrics['val_en_pr_auc']['macro_avg']:.4f} |",
    f"| zero-shot es toxic PR-AUC | "
    f"{train_metrics['eval_multilingual_toxic_pr_auc_by_lang']['es']['toxic']:.4f} |",
    f"| zero-shot it toxic PR-AUC | "
    f"{train_metrics['eval_multilingual_toxic_pr_auc_by_lang']['it']['toxic']:.4f} |",
    f"| zero-shot tr toxic PR-AUC | "
    f"{train_metrics['eval_multilingual_toxic_pr_auc_by_lang']['tr']['toxic']:.4f} |",
    f"| calibration temperature | {temperature:.4f} |",
    f"| ECE before calibration | {ece_before:.4f} |",
    f"| ECE after calibration | {ece_after:.4f} |",
    f"| fp32→int8 parity (macro PR-AUC delta) | {deltas.get('macro_avg', 0.0):+.4f} |",
    "",
    "## Known limitations",
    "",
    "- Japanese (ja): no training or eval data at all — zero-shot, unmeasured until "
    "live-traffic FPR measurement.",
    "- es/it/tr eval is `toxic` label only (Kaggle's `validation.csv` doesn't carry the "
    "other 5 labels) — no per-label multilingual breakdown.",
    "- Distillation degrades zero-shot cross-lingual transfer more than English "
    "quality: the KD objective only ever sees English batches.",
]
model_card = "\n".join(model_card_lines)
with open(EXPORT_DIR / "model_card.md", "w") as f:
    f.write(model_card + "\n")
print(model_card)

## Write metrics.json

In [ ]:
export_metrics = {
    "version": VERSION_TAG,
    "student_checkpoint": str(STUDENT_CHECKPOINT_DIR),
    "onnx_opset": ONNX_OPSET,
    "temperature": temperature,
    "ece_before": ece_before,
    "ece_after": ece_after,
    "onnx_fp32_size_mb": fp32_mb,
    "onnx_int8_size_mb": int8_mb,
    "parity_sample_size": len(parity_df),
    "parity_fp32_pr_auc": fp32_scores,
    "parity_int8_pr_auc": int8_scores,
    "parity_deltas": deltas,
    "parity_passed": parity_passed,
}
with open(EXPORT_DIR / "metrics.json", "w") as f:
    json.dump(export_metrics, f, indent=2)
print(f"wrote {EXPORT_DIR / 'metrics.json'}")

## Upload to R2

In [ ]:
import boto3
from google.colab import userdata

r2_secret_access_key = userdata.get("R2_SECRET_ACCESS_KEY")

s3 = boto3.client(
    "s3",
    endpoint_url=R2_ENDPOINT,
    aws_access_key_id=R2_ACCESS_KEY_ID,
    aws_secret_access_key=r2_secret_access_key,
)

version_prefix = f"{VERSION_TAG}/"

flat_files = [
    onnx_int8_path,
    EXPORT_DIR / "calibration.json",
    EXPORT_DIR / "thresholds.yaml",
    EXPORT_DIR / "model_card.md",
    EXPORT_DIR / "metrics.json",
]
for path in flat_files:
    key = f"{version_prefix}{path.name}"
    s3.upload_file(str(path), R2_BUCKET, key)
    print(f"uploaded {path.name} -> s3://{R2_BUCKET}/{key}")

for path in sorted(tokenizer_dir.iterdir()):
    key = f"{version_prefix}tokenizer/{path.name}"
    s3.upload_file(str(path), R2_BUCKET, key)
    print(f"uploaded tokenizer/{path.name} -> s3://{R2_BUCKET}/{key}")

print(f"\nall artifacts uploaded under s3://{R2_BUCKET}/{version_prefix}")